In [ ]:
import os
import numpy as np
import random
import math
import pydicom
from pydicom.dataset import Dataset, FileDataset
from datetime import datetime

In [ ]:
OUTPUT_DIR = "../data/ct_ellipses_new"
NUM_IMAGES = 400000
IMG_SIZE = (128, 128)  # (rows, cols)

NUM_ELLIPSES_RANGE = (1, 30)

AXIS_MIN = 1
AXIS_MAX = 64

HU_BACKGROUND = 0
HU_FILL_MIN = -500
HU_FILL_MAX = 1000

CLIP_MIN = -1000
CLIP_MAX = 3000

PIXEL_SPACING = [1.0, 1.0]

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)


def sample_axis_size(min_val=AXIS_MIN, max_val=AXIS_MAX):
    """
    Losuje rozmiar półosi z rozkładu o gęstości jednostajnie malejącej:
    p(x) = 2*(max-x) / (max-min)^2  dla x in [min, max]
    Gęstość wynosi 0 dla x=max i jest maksymalna dla x=min.
    Całkowanie odwrotne CDF: x = max - (max-min)*sqrt(U), U ~ Uniform(0,1)
    """
    u = random.random()
    x = max_val - (max_val - min_val) * math.sqrt(u)
    return max(min_val, int(round(x)))


def add_rotated_ellipse(image, center, axes, angle, value):
    """Dodaje wartość elipsy do obrazu (sumowanie zamiast nadpisywania)."""
    cy, cx = center
    a, b = axes
    theta = math.radians(angle)

    rows, cols = image.shape
    y, x = np.ogrid[:rows, :cols]

    x0 = x - cx
    y0 = y - cy

    xr = x0 * math.cos(theta) + y0 * math.sin(theta)
    yr = -x0 * math.sin(theta) + y0 * math.cos(theta)

    mask = (xr**2) / (a**2) + (yr**2) / (b**2) <= 1
    image[mask] += value

In [ ]:
from tqdm import tqdm

for i in tqdm(range(NUM_IMAGES), desc="Generating CT phantom slices", unit="slice"):
    img = np.full(IMG_SIZE, HU_BACKGROUND, dtype=np.float32)

    num_ellipses = random.randint(*NUM_ELLIPSES_RANGE)
    for _ in range(num_ellipses):
        center = (
            random.randint(0, IMG_SIZE[0] - 1),
            random.randint(0, IMG_SIZE[1] - 1),
        )
        
        axes = (sample_axis_size(), sample_axis_size())
        angle = random.uniform(0, 180)

        value = random.uniform(HU_FILL_MIN, HU_FILL_MAX)

        add_rotated_ellipse(img, center, axes, angle, value)

    img_min, img_max = img.min(), img.max()
    if img_min < CLIP_MIN or img_max > CLIP_MAX:
        img = (img - img_min) / (img_max - img_min) * (CLIP_MAX - CLIP_MIN) + CLIP_MIN

    img = img.astype(np.int16)

    # DICOM DATASET
    file_meta = Dataset()
    file_meta.MediaStorageSOPClassUID = pydicom.uid.CTImageStorage
    file_meta.MediaStorageSOPInstanceUID = pydicom.uid.generate_uid()
    file_meta.TransferSyntaxUID = pydicom.uid.ExplicitVRLittleEndian

    ds = FileDataset(
        filename_or_obj=None,
        dataset=Dataset(),
        file_meta=file_meta,
        preamble=b"\0" * 128,
    )

    # --- Identifiers ---
    ds.SOPClassUID = file_meta.MediaStorageSOPClassUID
    ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
    ds.StudyInstanceUID = pydicom.uid.generate_uid()
    ds.SeriesInstanceUID = pydicom.uid.generate_uid()

    ds.Modality = "CT"
    ds.ImageType = ["ORIGINAL", "PRIMARY", "AXIAL"]
    ds.PatientName = "PHANTOM^ELLIPSE"
    ds.PatientID = "000001"

    ds.StudyDate = datetime.now().strftime("%Y%m%d")
    ds.StudyTime = datetime.now().strftime("%H%M%S")

    # --- Geometry ---
    ds.Rows, ds.Columns = IMG_SIZE
    ds.PixelSpacing = PIXEL_SPACING
    ds.SliceThickness = 1.0
    ds.ImagePositionPatient = [0.0, 0.0, float(i)]
    ds.ImageOrientationPatient = [1, 0, 0, 0, 1, 0]
    ds.ReconstructionDiameter = max(IMG_SIZE) * PIXEL_SPACING[0]

    # --- Pixel data ---
    ds.BitsAllocated = 16
    ds.BitsStored = 16
    ds.HighBit = 15
    ds.PixelRepresentation = 1  # signed int
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"

    # HU scaling
    ds.RescaleIntercept = 0
    ds.RescaleSlope = 1

    ds.PixelData = img.tobytes()

    filename = os.path.join(OUTPUT_DIR, f"CT_{i:06d}.dcm")
    ds.save_as(filename)

print("DICOM CT dataset wygenerowany:", OUTPUT_DIR)